In [1]:
import pandas as pd
import requests
from io import StringIO
from functools import reduce
import time 


Sito per trovare i dataflow corretti rispetto a quello che fornisce ISTAT nell'api: https://datiistat.it/alldataflows 

Banca dati ISTAT: https://esploradati.istat.it/databrowser/#/it/dw/categories


In [2]:
start = time.time()

# start_date = '2013-01'
start_date = '2004-01'
end_date = '2026-01'

start_year = 2004

# 1. Numero Occupati

In [3]:
# API endpoint and dataset
base = "https://esploradati.istat.it/SDMXWS/rest/data"
# dataflow = "IT1,150_875"
dataflow = "IT1,150_875_DF_DCCV_OCCUPATIMENS1_1"

# Query parameters
params = {
    "startPeriod": f"{start_date}",
    "endPeriod": f"{end_date}",
    #"POSIZ_PROF": "1", #Dipedenti
    #"PERM_TEMP_EMPLOYEES": ""
    "format": "CSV"  # request CSV
}

# Headers (fixed syntax)
headers = {"Accept": "text/csv"}

# Build URL
url = f"{base}/{dataflow}"

# Send GET request
response = requests.get(url, params=params, headers=headers, verify=False)

# Check for errors
if response.status_code != 200:
    raise Exception(f"Error {response.status_code}: {response.text}")

# Read CSV into pandas directly
csv_data = StringIO(response.text)
df = pd.read_csv(csv_data)


c:\Users\francesco.demo\Desktop\Personal Repos\Employment_rate_predictor\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'esploradati.istat.it'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\francesco.demo\AppData\Local\Temp\ipykernel_10900\2975726017.py:30: DtypeWarning: Columns (0: OBS_STATUS, 1: NOTE_DATA_TYPE) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_data)


In [4]:
#"POSIZ_PROF"--> #'posizione professionale': # 1: Dipedenti, # 2: Indipendenti,# 9: Totale
#"PERM_TEMP_EMPLOYEES" --> 'Categoria occupazione dipendente' #1: Tempo determinato, 2: Tempo indeterminato,9: Totale

last_edition = df['EDITION'].unique().max()
df = df[df['EDITION'] == last_edition]

df = df[(df['ADJUSTMENT'] == 'N') #prendo  solo le righe che NON hanno adjustments
      & (df['POSIZ_PROF']==1)  # prendo solo i lavoratori Dipendenti
      & (df['PERM_TEMP_EMPLOYEES']!=9)] # prendo solo il valore totale

# df['PERM_TEMP_EMPLOYEES'].replace({1:'fixed_term', 2 : 'permanent'}, inplace = True)
df['PERM_TEMP_EMPLOYEES'] = df['PERM_TEMP_EMPLOYEES'].replace({1:'fixed_term', 2 : 'permanent'})

In [5]:
#"POSIZ_PROF"--> #'posizione professionale': # 1: Dipedenti, # 2: Indipendenti,# 9: Totale
#"PERM_TEMP_EMPLOYEES" --> 'Categoria occupazione dipendente' #1: Tempo determinato, 2: Tempo indeterminato,9: Totale

last_edition = df['EDITION'].unique().max()
df = df[df['EDITION'] == last_edition]

df = df[(df['ADJUSTMENT'] == 'N') #prendo  solo le righe che NON hanno adjustments
      & (df['POSIZ_PROF']==1)  # prendo solo i lavoratori Dipendenti
      & (df['PERM_TEMP_EMPLOYEES']!=9)] # prendo solo il valore totale

df['PERM_TEMP_EMPLOYEES'].replace({1:'fixed_term', 2 : 'permanent'}, inplace = True)
df_employment_pivoted = df.pivot(index='TIME_PERIOD', columns='PERM_TEMP_EMPLOYEES', values=['OBS_VALUE'])

df_employment_pivoted = df_employment_pivoted.rename_axis(None)

df_employment_pivoted.columns = df_employment_pivoted.columns.droplevel(0)

df_employment_pivoted['date'] =  pd.to_datetime(df_employment_pivoted.index, format='%Y-%m')

df_employment_pivoted = df_employment_pivoted.rename_axis(None, axis=1)

df_employment_pivoted = df_employment_pivoted.reset_index()#
df_employment_pivoted.drop('index', axis = 1, inplace = True)#

C:\Users\francesco.demo\AppData\Local\Temp\ipykernel_10900\2685154527.py:11: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['PERM_TEMP_EMPLOYEES'].replace({1:'fixed_term', 2 : 'permanent'}, inplace = True)


# 2. Tasso di occupazione

In [6]:
# API endpoint and dataset
base = "https://esploradati.istat.it/SDMXWS/rest/data"
# dataflow = "IT1,150_875"
dataflow = "IT1,150_872_DF_DCCV_TAXOCCUMENS1_1"

# Query parameters
params = {
    "startPeriod": f"{start_date}",
    "endPeriod": f"{end_date}",
    #"POSIZ_PROF": "1", #Dipedenti
    #"PERM_TEMP_EMPLOYEES": ""
    "format": "CSV"  # request CSV
}

# Headers (fixed syntax)
headers = {"Accept": "text/csv"}

# Build URL
url = f"{base}/{dataflow}"

# Send GET request
response = requests.get(url, params=params, headers=headers, verify=False)

# Check for errors
if response.status_code != 200:
    raise Exception(f"Error {response.status_code}: {response.text}")

# Read CSV into pandas directly
csv_data = StringIO(response.text)
df = pd.read_csv(csv_data)


c:\Users\francesco.demo\Desktop\Personal Repos\Employment_rate_predictor\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'esploradati.istat.it'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\francesco.demo\AppData\Local\Temp\ipykernel_10900\2500150188.py:30: DtypeWarning: Columns (0: OBS_STATUS) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_data)


In [7]:
last_edition = df['EDITION'].unique().max() # this is necessary to take the most updated values
df = df[df['EDITION'] == last_edition]

df_employed_rate = df[(df.SEX==9) & (df.AGE.isin(['Y15-64','Y15-34'])) & (df.ADJUSTMENT == 'N')] # I select youth employment and full age range employment

df_employed_rate_pivoted = df_employed_rate.pivot(index='TIME_PERIOD', columns='AGE', values=['OBS_VALUE'])

df_employed_rate_pivoted = df_employed_rate_pivoted.rename_axis(None)

df_employed_rate_pivoted.columns = df_employed_rate_pivoted.columns.droplevel(0)

df_employed_rate_pivoted['date'] =  pd.to_datetime(df_employed_rate_pivoted.index, format='%Y-%m')

df_employed_rate_pivoted = df_employed_rate_pivoted.rename_axis(None, axis=1)

df_employed_rate_pivoted.rename(columns={'Y15-64': 'employment rate (Y15-64)', 'Y15-34':  'youth employment rate (Y15-34)'}, inplace = True)

df_employed_rate_pivoted = df_employed_rate_pivoted.reset_index()#
df_employed_rate_pivoted.drop('index', axis = 1, inplace = True)#


# 3. Forza lavoro

In [8]:
# API endpoint and dataset
base = "https://esploradati.istat.it/SDMXWS/rest/data"
# dataflow = "IT1,150_875"
dataflow = "IT1,150_873_DF_DCCV_FORZLVMENS1_1"

# Query parameters
params = {
    "startPeriod": f"{start_date}",
    "endPeriod": f"{end_date}",
    #"POSIZ_PROF": "1", #Dipedenti
    #"PERM_TEMP_EMPLOYEES": ""
    "format": "CSV"  # request CSV
}

# Headers (fixed syntax)
headers = {"Accept": "text/csv"}

# Build URL
url = f"{base}/{dataflow}"

# Send GET request
response = requests.get(url, params=params, headers=headers, verify=False)

# Check for errors
if response.status_code != 200:
    raise Exception(f"Error {response.status_code}: {response.text}")

# Read CSV into pandas directly
csv_data = StringIO(response.text)
df = pd.read_csv(csv_data)


c:\Users\francesco.demo\Desktop\Personal Repos\Employment_rate_predictor\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'esploradati.istat.it'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\francesco.demo\AppData\Local\Temp\ipykernel_10900\795334971.py:30: DtypeWarning: Columns (0: OBS_STATUS) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_data)


In [9]:
last_edition = df['EDITION'].unique().max() # this is necessary to take the most updated values
df = df[df['EDITION'] == last_edition]

df_workforce = df[(df.SEX==9) & (df.AGE=='Y15-64') & (df.ADJUSTMENT == 'N')]

df_workforce_pivoted = df_workforce.pivot(index='TIME_PERIOD', columns='AGE', values=['OBS_VALUE'])

df_workforce_pivoted = df_workforce_pivoted.rename_axis(None)

df_workforce_pivoted.columns = df_workforce_pivoted.columns.droplevel(0)

df_workforce_pivoted['date'] =  pd.to_datetime(df_workforce_pivoted.index, format='%Y-%m')

df_workforce_pivoted = df_workforce_pivoted.rename_axis(None, axis=1)

df_workforce_pivoted.rename(columns={'Y15-64': 'workforce (Y15-64)'}, inplace = True)

df_workforce_pivoted = df_workforce_pivoted.reset_index()#
df_workforce_pivoted.drop('index', axis = 1, inplace = True)#


# 4. Tasso di Attività

In [10]:
# API endpoint and dataset
base = "https://esploradati.istat.it/SDMXWS/rest/data"
# dataflow = "IT1,150_875"
dataflow = "IT1,150_876_DF_DCCV_TAXATVTMENS1_1"

# Query parameters
params = {
    "startPeriod": f"{start_date}",
    "endPeriod": f"{end_date}",
    "format": "CSV"  # request CSV
}

# Headers (fixed syntax)
headers = {"Accept": "text/csv"}

# Build URL
url = f"{base}/{dataflow}"

# Send GET request
response = requests.get(url, params=params, headers=headers, verify=False)

# Check for errors
if response.status_code != 200:
    raise Exception(f"Error {response.status_code}: {response.text}")

# Read CSV into pandas directly
csv_data = StringIO(response.text)
df = pd.read_csv(csv_data)


c:\Users\francesco.demo\Desktop\Personal Repos\Employment_rate_predictor\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'esploradati.istat.it'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\francesco.demo\AppData\Local\Temp\ipykernel_10900\2316900606.py:28: DtypeWarning: Columns (0: OBS_STATUS) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_data)


In [11]:
last_edition = df['EDITION'].unique().max() # this is necessary to take the most updated values
df = df[df['EDITION'] == last_edition]

df_activity_rate = df[(df.SEX==9) & (df.AGE=='Y15-64') & (df.ADJUSTMENT == 'N')]

df_activity_rate_pivoted = df_activity_rate.pivot(index='TIME_PERIOD', columns='AGE', values=['OBS_VALUE'])

df_activity_rate_pivoted = df_activity_rate_pivoted.rename_axis(None)

df_activity_rate_pivoted.columns = df_activity_rate_pivoted.columns.droplevel(0)

df_activity_rate_pivoted['date'] =  pd.to_datetime(df_activity_rate_pivoted.index, format='%Y-%m')

df_activity_rate_pivoted = df_activity_rate_pivoted.rename_axis(None, axis=1)

df_activity_rate_pivoted.rename(columns={'Y15-64': 'activity rate (Y15-64)'}, inplace =True)

df_activity_rate_pivoted = df_activity_rate_pivoted.reset_index()#
df_activity_rate_pivoted.drop('index', axis = 1, inplace = True)#

# 5. Tasso di disoccupazione

In [12]:
# API endpoint and dataset
base = "https://esploradati.istat.it/SDMXWS/rest/data"
# dataflow = "IT1,150_875"
dataflow = "IT1,151_874_DF_DCCV_TAXDISOCCUMENS1_1"

# Query parameters
params = {
    "startPeriod": f"{start_date}",
    "endPeriod": f"{end_date}",
    "format": "CSV"  # request CSV
}

# Headers (fixed syntax)
headers = {"Accept": "text/csv"}

# Build URL
url = f"{base}/{dataflow}"

# Send GET request
response = requests.get(url, params=params, headers=headers, verify=False)

# Check for errors
if response.status_code != 200:
    raise Exception(f"Error {response.status_code}: {response.text}")

# Read CSV into pandas directly
csv_data = StringIO(response.text)
df = pd.read_csv(csv_data)


c:\Users\francesco.demo\Desktop\Personal Repos\Employment_rate_predictor\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'esploradati.istat.it'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\francesco.demo\AppData\Local\Temp\ipykernel_10900\2850697168.py:28: DtypeWarning: Columns (0: OBS_STATUS) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_data)


In [ ]:
last_edition = df['EDITION'].unique().max() # this is necessary to take the most updated values
df = df[df['EDITION'] == last_edition]

df_unemployment_rate = df[(df.SEX==9) & (df.AGE=='Y15-64') & (df.ADJUSTMENT == 'N')]

df_unemployment_rate_pivoted = df_unemployment_rate.pivot(index='TIME_PERIOD', columns='AGE', values=['OBS_VALUE'])

df_unemployment_rate_pivoted = df_unemployment_rate_pivoted.rename_axis(None)

df_unemployment_rate_pivoted.columns = df_unemployment_rate_pivoted.columns.droplevel(0)

df_unemployment_rate_pivoted['date'] =  pd.to_datetime(df_unemployment_rate_pivoted.index, format='%Y-%m')

df_unemployment_rate_pivoted = df_unemployment_rate_pivoted.rename_axis(None, axis=1)

df_unemployment_rate_pivoted.rename(columns={'Y15-64': 'unemployment rate (Y15-64)'}, inplace = True)

df_unemployment_rate_pivoted = df_unemployment_rate_pivoted.reset_index()#

df_unemployment_rate_pivoted.drop('index', axis = 1, inplace = True)#

# 6. Inattivi

In [14]:
# API endpoint and dataset
base = "https://esploradati.istat.it/SDMXWS/rest/data"
# dataflow = "IT1,150_875"
dataflow = "IT1,152_879_DF_DCCV_INATTIVMENS1_1"

# Query parameters
params = {
    "startPeriod": f"{start_date}",
    "endPeriod": f"{end_date}",
    "format": "CSV"  # request CSV
}

# Headers (fixed syntax)
headers = {"Accept": "text/csv"}

# Build URL
url = f"{base}/{dataflow}"

# Send GET request
response = requests.get(url, params=params, headers=headers, verify=False)

# Check for errors
if response.status_code != 200:
    raise Exception(f"Error {response.status_code}: {response.text}")

# Read CSV into pandas directly
csv_data = StringIO(response.text)
df = pd.read_csv(csv_data)


c:\Users\francesco.demo\Desktop\Personal Repos\Employment_rate_predictor\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'esploradati.istat.it'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\francesco.demo\AppData\Local\Temp\ipykernel_10900\1700672484.py:28: DtypeWarning: Columns (0: OBS_STATUS) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_data)


In [15]:
last_edition = df['EDITION'].unique().max() # this is necessary to take the most updated values
df = df[df['EDITION'] == last_edition]

df_inactives = df[(df.SEX==9) & (df.AGE=='Y15-64') & (df.ADJUSTMENT == 'N')]

df_inactives_pivoted = df_inactives.pivot(index='TIME_PERIOD', columns='AGE', values=['OBS_VALUE'])

df_inactives_pivoted = df_inactives_pivoted.rename_axis(None)

df_inactives_pivoted.columns = df_inactives_pivoted.columns.droplevel(0)

df_inactives_pivoted['date'] =  pd.to_datetime(df_inactives_pivoted.index, format='%Y-%m')

df_inactives_pivoted = df_inactives_pivoted.rename_axis(None, axis=1)

df_inactives_pivoted.rename(columns={'Y15-64': 'inactives (Y15-64)'}, inplace = True)

df_inactives_pivoted = df_inactives_pivoted.reset_index()#

df_inactives_pivoted.drop('index', axis = 1, inplace = True)#


# 7. Prezzi della produzione dell'industria

In [16]:
# API endpoint and dataset
base = "https://esploradati.istat.it/SDMXWS/rest/data"
# dataflow = "IT1,150_875"
dataflow = "IT1,145_360_DF_DCSC_PREZZPIND_1_4"

# Query parameters
params = {
    "startPeriod": f"{start_date}",
    "endPeriod": f"{end_date}",
    "format": "CSV"  # request CSV
}

# Headers (fixed syntax)
headers = {"Accept": "text/csv"}

# Build URL
url = f"{base}/{dataflow}"

# Send GET request
response = requests.get(url, params=params, headers=headers, verify=False)

# Check for errors
if response.status_code != 200:
    raise Exception(f"Error {response.status_code}: {response.text}")

# Read CSV into pandas directly
csv_data = StringIO(response.text)
df = pd.read_csv(csv_data)

c:\Users\francesco.demo\Desktop\Personal Repos\Employment_rate_predictor\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'esploradati.istat.it'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\francesco.demo\AppData\Local\Temp\ipykernel_10900\3374446059.py:28: DtypeWarning: Columns (0: ECON_ACTIVITY_NACE_2007, 1: NOTE_MARKET) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_data)


In [17]:
df_industry_production_prices = df[(df['MARKET']=='T')  & (df['ECON_ACTIVITY_NACE_2007']=='0020')] [['TIME_PERIOD', 'OBS_VALUE']].reset_index().rename(columns={'TIME_PERIOD':'date', 'OBS_VALUE':'industry_production_prices'})
df_industry_production_prices['date'] =  pd.to_datetime(df_industry_production_prices['date'], format='%Y-%m')#
df_industry_production_prices.drop('index', axis=1, inplace= True)

# 8. Prezzi della produzione dei servizi

In [18]:
# API endpoint and dataset
base = "https://esploradati.istat.it/SDMXWS/rest/data"
# dataflow = "IT1,150_875"
dataflow = "IT1,145_376_DF_DCSC_PREZPRODSERV_1_7"

# Query parameters
params = {
    "startPeriod": f"{start_date}",
    "endPeriod": f"{end_date}",
    "format": "CSV"  # request CSV
}

# Headers (fixed syntax)
headers = {"Accept": "text/csv"}

# Build URL
url = f"{base}/{dataflow}"

# Send GET request
response = requests.get(url, params=params, headers=headers, verify=False)

# Check for errors
if response.status_code != 200:
    raise Exception(f"Error {response.status_code}: {response.text}")

# Read CSV into pandas directly
csv_data = StringIO(response.text)
df = pd.read_csv(csv_data)

c:\Users\francesco.demo\Desktop\Personal Repos\Employment_rate_predictor\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'esploradati.istat.it'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [19]:
df_services_production_prices = df[(df['ECON_ACTIVITY_NACE_2007']=='H_N_XK')][['TIME_PERIOD','OBS_VALUE']].reset_index().rename(columns={'TIME_PERIOD':'date', 'OBS_VALUE':'services_production_prices'})
df_services_production_prices['date'] = df_services_production_prices['date'].apply(lambda x: pd.Period(x, freq="Q").to_timestamp(how="start")).dt.strftime("%Y-%m-%d")
df_services_production_prices['date'] =  pd.to_datetime(df_services_production_prices['date'], format='%Y-%m-%d')
df_services_production_prices.drop('index', axis=1, inplace= True)

# 9. Fiducia nelle imprese

In [20]:
# API endpoint and dataset
base = "https://esploradati.istat.it/SDMXWS/rest/data"
# dataflow = "IT1,150_875"
dataflow = "IT1,6_64_DF_DCSC_IESI_2"

# Query parameters
params = {
    "startPeriod": f"{start_date}",
    "endPeriod": f"{end_date}",
    "format": "CSV"  # request CSV
}

# Headers (fixed syntax)
headers = {"Accept": "text/csv"}

# Build URL
url = f"{base}/{dataflow}"

# Send GET request
response = requests.get(url, params=params, headers=headers, verify=False)

# Check for errors
if response.status_code != 200:
    raise Exception(f"Error {response.status_code}: {response.text}")

# Read CSV into pandas directly
csv_data = StringIO(response.text)
df = pd.read_csv(csv_data)



c:\Users\francesco.demo\Desktop\Personal Repos\Employment_rate_predictor\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'esploradati.istat.it'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [21]:
df_companies_trust = df[['TIME_PERIOD','OBS_VALUE']].reset_index().rename(columns={'TIME_PERIOD':'date', 'OBS_VALUE':'companies_trust'})
df_companies_trust['date'] =  pd.to_datetime(df_companies_trust['date'], format='%Y-%m')
df_companies_trust.drop('index', axis=1, inplace= True)

# 10. Prezzo Benzina

In [22]:
df = pd.read_csv('https://sisen.mase.gov.it/dgsaie/api/v1/monthly-prices/1/export?format=CSV&lang=it' )
df["date"] = pd.to_datetime(df["ANNO"].astype(str) + "-" + df["CODICE_MESE"].astype(str) + "-01",  format="%Y-%m-%d")
df["gasoline_car_price"] = df["PREZZO"].str.replace(",", "", regex=False)
df["gasoline_car_price"] = df["gasoline_car_price"].astype('float')
df_gasoline_car_price = df[df['ANNO'] >= start_year][['date', 'gasoline_car_price']].reset_index()
df_gasoline_car_price.drop('index', axis=1, inplace=True)


# 11. Prezzo Gasolio

In [23]:
df = pd.read_csv('https://sisen.mase.gov.it/dgsaie/api/v1/monthly-prices/2/export?format=CSV&lang=it' )
df["date"] = pd.to_datetime(df["ANNO"].astype(str) + "-" + df["CODICE_MESE"].astype(str) + "-01",  format="%Y-%m-%d")
df["diesel_price"] = df["PREZZO"].str.replace(",", "", regex=False)
df["diesel_price"] = df["diesel_price"].astype('float')
df_diesel_price = df[df['ANNO'] >= start_year ][['date', 'diesel_price']].reset_index()
df_diesel_price.drop('index', axis=1, inplace=True)


# 12. Prezzo GPL

In [24]:
df = pd.read_csv('https://sisen.mase.gov.it/dgsaie/api/v1/monthly-prices/5/export?format=CSV&lang=it' )
df["date"] = pd.to_datetime(df["ANNO"].astype(str) + "-" + df["CODICE_MESE"].astype(str) + "-01",  format="%Y-%m-%d")
df["PREZZO"] = df["PREZZO"].str.replace(",", "", regex=False)
df["GPL_price"] =str("0.") + df["PREZZO"].astype('str')
df["GPL_price"] = df["GPL_price"].astype('float')
df_GPL_price = df[df['ANNO'] >= start_year ][['date', 'GPL_price']].reset_index()
df_GPL_price.drop('index', axis=1, inplace=True)


# 13. Prezzo Gasolio riscaldamento

In [25]:
df = pd.read_csv('https://sisen.mase.gov.it/dgsaie/api/v1/monthly-prices/3/export?format=CSV&lang=it' )
df["date"] = pd.to_datetime(df["ANNO"].astype(str) + "-" + df["CODICE_MESE"].astype(str) + "-01",  format="%Y-%m-%d")
df["diesel_heatining_price"] = df["PREZZO"].str.replace(",", "", regex=False)
df["diesel_heatining_price"] = df["diesel_heatining_price"].astype('float')
df_diesel_heatining_price = df[df['ANNO'] >= start_year][['date', 'diesel_heatining_price']].reset_index()
df_diesel_heatining_price.drop('index', axis=1, inplace=True)


# Genero il dataset finale

In [26]:
dfs = [df_employment_pivoted,
        df_employed_rate_pivoted,
        df_workforce_pivoted,
        df_activity_rate_pivoted,
        df_unemployment_rate_pivoted,
        df_inactives_pivoted,
        df_industry_production_prices,
        df_services_production_prices, 
        df_gasoline_car_price,
        df_diesel_price,
        df_GPL_price,
        df_diesel_heatining_price,
        df_companies_trust]

In [27]:
df1 = pd.merge(df_employment_pivoted, df_employed_rate_pivoted, on='date', how='outer')

df2 = pd.merge(df1, df_workforce_pivoted, on='date', how='outer')

df3 = pd.merge(df2, df_activity_rate_pivoted, on='date', how='outer')

df4 = pd.merge(df3, df_unemployment_rate_pivoted, on='date', how='outer')

df5 = pd.merge(df4, df_inactives_pivoted, on='date', how='outer')

df6 = pd.merge(df5, df_industry_production_prices, on='date', how='outer')

df7 = pd.merge(df6, df_services_production_prices, on='date', how='outer')

df8 = pd.merge(df7, df_gasoline_car_price, on='date', how='inner')

df9 = pd.merge(df8, df_diesel_price, on='date', how='inner')

df10 = pd.merge(df9, df_GPL_price, on='date', how='inner')

df11 = pd.merge(df10, df_diesel_heatining_price, on='date', how='inner')

final_df = pd.merge(df11, df_companies_trust, on='date', how='inner')

final_df

,fixed_term,permanent,date,youth employment rate (Y15-34),employment rate (Y15-64),workforce (Y15-64),activity rate (Y15-64),unemployment rate (Y15-64),inactives (Y15-64),industry_production_prices,services_production_prices,gasoline_car_price,diesel_price,GPL_price,diesel_heatining_price,companies_trust
0,2034.168,14303.085,2005-03-01,51.199650,57.129196,23913.712,62.340694,8.359706,14446.002,78.1,NaN,1.17015,1.05744,0.55297,1.00754,93.5
1,2082.989,14185.766,2005-04-01,51.599862,57.365899,23905.030,62.297724,7.916542,14467.207,78.3,NaN,1.21972,1.10630,0.55373,1.02208,93.0
2,2060.206,14365.222,2005-05-01,51.491946,57.617419,23892.852,62.246996,7.437431,14491.091,78.2,NaN,1.20874,1.07075,0.55322,9974.00000,92.2
3,2021.367,14516.212,2005-06-01,51.785177,57.664283,23955.620,62.413888,7.609853,14426.254,78.6,NaN,1.20900,1.09044,0.55320,1.03456,89.9
4,2194.322,14267.287,2005-07-01,51.892967,57.192030,23506.991,61.246143,6.619378,14874.186,78.8,NaN,1.24486,1.13104,0.55342,1.06796,92.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253,2649.801,16443.037,2026-04-01,45.158319,63.803318,24974.184,67.283592,5.172545,12143.608,130.3,118.5,1.76366,2.09789,0.79194,1.94356,95.3
254,2541.964,16611.265,2026-05-01,44.543625,63.553815,24975.048,67.298528,5.564332,12135.791,130.0,NaN,1.93113,2.00699,0.82563,1.81046,94.3
255,2669.426,16332.159,2026-06-01,44.419068,62.874138,25260.583,68.079365,7.645822,11844.027,130.0,NaN,1.86567,1.95353,0.78777,1.65882,95.3
256,2559.650,16553.336,2026-07-01,43.955166,63.360042,25116.056,67.766504,6.502418,11946.585,133.0,NaN,1.90820,2.02701,0.76301,1.77020,95.7


In [28]:
print('Finito')

Finito


In [29]:
final_df.rename(columns = {'workforce (Y15-64)': 'workforce', 
                           'inactives (Y15-64)': 'inactives',
                            'youth employment rate (Y15-34)': 'youth_employment_rate', 
                            # 'employment rate (Y15-34)': 'employment_rate', 
                            'employment rate (Y15-64)':'employment_rate', 
                            # 'youth employment rate (Y15-64)':'youth_employment_rate', 
                            'unemployment rate (Y15-64)': 'unemployment_rate', 
                            'activity rate (Y15-64)': 'activity_rate' }, inplace =True)


final_df = final_df[['date', #'fixed_term', 'permanent', 
                     'workforce', 'inactives',
                    'employment_rate', 'youth_employment_rate', 'activity_rate', 'unemployment_rate', 
                    'industry_production_prices', 'services_production_prices', 'companies_trust', 
                    'gasoline_car_price', 'diesel_price', 'GPL_price', 'diesel_heatining_price']]

In [30]:
final_df[['employment_rate', 'youth_employment_rate', 
          'activity_rate', 'unemployment_rate']] = final_df[['employment_rate', 'youth_employment_rate', 'activity_rate', 'unemployment_rate']].round(2)

final_df

,date,workforce,inactives,employment_rate,youth_employment_rate,activity_rate,unemployment_rate,industry_production_prices,services_production_prices,companies_trust,gasoline_car_price,diesel_price,GPL_price,diesel_heatining_price
0,2005-03-01,23913.712,14446.002,57.13,51.20,62.34,8.36,78.1,NaN,93.5,1.17015,1.05744,0.55297,1.00754
1,2005-04-01,23905.030,14467.207,57.37,51.60,62.30,7.92,78.3,NaN,93.0,1.21972,1.10630,0.55373,1.02208
2,2005-05-01,23892.852,14491.091,57.62,51.49,62.25,7.44,78.2,NaN,92.2,1.20874,1.07075,0.55322,9974.00000
3,2005-06-01,23955.620,14426.254,57.66,51.79,62.41,7.61,78.6,NaN,89.9,1.20900,1.09044,0.55320,1.03456
4,2005-07-01,23506.991,14874.186,57.19,51.89,61.25,6.62,78.8,NaN,92.2,1.24486,1.13104,0.55342,1.06796
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253,2026-04-01,24974.184,12143.608,63.80,45.16,67.28,5.17,130.3,118.5,95.3,1.76366,2.09789,0.79194,1.94356
254,2026-05-01,24975.048,12135.791,63.55,44.54,67.30,5.56,130.0,NaN,94.3,1.93113,2.00699,0.82563,1.81046
255,2026-06-01,25260.583,11844.027,62.87,44.42,68.08,7.65,130.0,NaN,95.3,1.86567,1.95353,0.78777,1.65882
256,2026-07-01,25116.056,11946.585,63.36,43.96,67.77,6.50,133.0,NaN,95.7,1.90820,2.02701,0.76301,1.77020


In [34]:
final_df.isnull().sum()

date                            0
workforce                       1
inactives                       1
employment_rate                 1
youth_employment_rate           1
activity_rate                   1
unemployment_rate               1
industry_production_prices      0
services_production_prices    192
companies_trust                 1
gasoline_car_price              0
diesel_price                    0
GPL_price                       0
diesel_heatining_price          0
dtype: int64

In [32]:
final_df.to_csv('dataset_forecast.csv', index = False)

In [33]:
duration = time.time() - start
print(duration/60)

1.8435855269432069
